# POSTECH – BB | Tech Challenge Fase 3
## Análise de churn bancário
Objetivo: homologar consultas SQL no Colab antes da implementação em S3 e Athena.

**Etapa inicial:** carregar e validar a base. Execute as células em ordem.

## 1. Ambiente

In [ ]:
%pip -q install duckdb

In [ ]:
import pandas as pd
import duckdb
from google.colab import files
from IPython.display import display
print('DuckDB:', duckdb.__version__)

## 2. Carregamento local
Selecione o CSV original. Não publique dados bancários no GitHub sem autorização.

In [ ]:
from io import BytesIO
uploaded = files.upload()
filename = 'Bank Customer Churn Prediction.csv'
assert filename in uploaded, 'Selecione o arquivo original com o nome indicado.'
df = pd.read_csv(BytesIO(uploaded[filename]))
print('Dimensões:', df.shape)
display(df.head())

## 3. Validação da qualidade

In [ ]:
expected = ['customer_id','credit_score','country','gender','age','tenure','balance','products_number','credit_card','active_member','estimated_salary','churn']
assert list(df.columns) == expected, 'Estrutura inesperada'
display(pd.DataFrame({'tipo': df.dtypes.astype(str), 'nulos': df.isna().sum(), 'distintos': df.nunique()}))
print('Duplicatas:', df.duplicated().sum())
print('IDs duplicados:', df.customer_id.duplicated().sum())
for col in ['churn','active_member','credit_card']:
    assert df[col].isin([0,1]).all(), f'Valores inválidos: {col}'
assert df.isna().sum().sum() == 0
assert df.customer_id.is_unique

## 4. Conexão SQL e teste inicial
A tabela existe apenas na sessão atual do Colab.

In [ ]:
con = duckdb.connect(':memory:')
con.register('source_df', df)
con.execute('CREATE TABLE customers AS SELECT * FROM source_df')
display(con.sql('SELECT COUNT(*) AS total_clientes, SUM(churn) AS total_churn, ROUND(100.0 * AVG(churn), 2) AS taxa_churn_pct FROM customers').df())

## 5. Ponto de validação
Esperado para o arquivo fornecido: 10.000 registros, 12 colunas, 0 nulos, 0 duplicatas, 2.037 saídas (20,37%).

Após validação, acrescentaremos as dez consultas analíticas, gráficos e interpretações. Não execute etapas AWS ainda.